# FacTora — Multimodal Image Deepfake & AI Generation Detector Training
**Project**: FacTora Multimodal Credibility Intelligence Platform  
**Author**: Pranav Lamkhade  
**Architecture**: Vision Transformer / ResNet / ConvNeXt for Image Classification  
**Task**: Binary Classification — `REAL (0)` vs `AI-GENERATED / FAKE (1)`  
**Benchmark Dataset**: CIFAKE (120,000 images: 60k Real from CIFAR-10, 60k Synthetic from Stable Diffusion)  
**Target Accuracy**: >95.0% Validation Accuracy  
**Output**: Lightweight `model.safetensors` (~20-80 MB) for on-demand inference in FacTora.

In [ ]:
# Step 1: Environment & GPU Hardware Check
import torch
import torchvision
import transformers

print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16/FP16 Supported  : {torch.cuda.is_bf16_supported()}")
print(f"Transformers Version : {transformers.__version__}")

In [ ]:
# Step 2: Dependencies Installation
!pip install -q evaluate datasets accelerate safetensors

In [ ]:
# Step 3: Imports & Configuration
import os
import glob
import json
import numpy as np
from PIL import Image
import evaluate
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# ── Training Configuration ──────────────────────────────────────────────────
BASE_MODEL_NAME = "microsoft/resnet-50"
OUTPUT_DIR = "/kaggle/working/factora_image_model"
BATCH_SIZE = 128  # High throughput on Tesla T4 16GB
NUM_EPOCHS = 3    # Achieves >96% accuracy rapidly on CIFAKE
LEARNING_RATE = 1e-4
IMAGE_SIZE = (128, 128)  # Optimal balance between detail and training velocity
ID2LABEL = {0: "REAL", 1: "FAKE"}
LABEL2ID = {"REAL": 0, "FAKE": 1}

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Step 4: Locate Dataset on Kaggle
import os
import glob

print("--- Scanning /kaggle/input ---")
for item in os.listdir("/kaggle/input"):
    print(f"  - /kaggle/input/{item}")

DATASET_PATH = None
# Walk input directory to find the directory that directly contains 'train' and 'test'
for root, dirs, files in os.walk("/kaggle/input"):
    dirs_lower = [d.lower() for d in dirs]
    if "train" in dirs_lower and "test" in dirs_lower:
        DATASET_PATH = root
        break

if DATASET_PATH:
    print(f"\n[FOUND DATASET] Successfully located CIFAKE at: {DATASET_PATH}")
else:
    print("\n[WARNING] Dataset not found via auto-scan. Defaulting to /kaggle/input/cifake-real-and-ai-generated-synthetic-images")
    DATASET_PATH = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"


In [ ]:
# Step 5: PyTorch Dataset Loader with Resilient Path Resolution
class CIFAKEDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        
        # Find the split directory (case-insensitive)
        split_dir = None
        if os.path.exists(root_dir):
            for d in os.listdir(root_dir):
                if d.lower() == split.lower():
                    split_dir = os.path.join(root_dir, d)
                    break
        if not split_dir:
            split_dir = os.path.join(root_dir, split)
            
        real_dir = None
        fake_dir = None
        if os.path.exists(split_dir):
            for sub in os.listdir(split_dir):
                if sub.upper() == "REAL":
                    real_dir = os.path.join(split_dir, sub)
                elif sub.upper() == "FAKE":
                    fake_dir = os.path.join(split_dir, sub)
                    
        if real_dir and os.path.exists(real_dir):
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                for f in glob.glob(os.path.join(real_dir, ext)):
                    self.image_paths.append(f)
                    self.labels.append(0)  # REAL
                    
        if fake_dir and os.path.exists(fake_dir):
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                for f in glob.glob(os.path.join(fake_dir, ext)):
                    self.image_paths.append(f)
                    self.labels.append(1)  # FAKE
                    
        print(f"Loaded {len(self.image_paths)} samples for split: {split} (Real: {self.labels.count(0)}, Fake: {self.labels.count(1)})")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "labels": torch.tensor(label, dtype=torch.long)}

# Image Preprocessor transforms
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CIFAKEDataset(DATASET_PATH, split="train", transform=train_transforms)
val_dataset = CIFAKEDataset(DATASET_PATH, split="test", transform=val_transforms)


In [ ]:
# Step 6: Load Image Processor and Pretrained Model
processor = AutoImageProcessor.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture : {BASE_MODEL_NAME}")
print(f"Total Parameters   : {total_params / 1e6:.2f} M")
print(f"Trainable Params   : {trainable_params / 1e6:.2f} M")

In [ ]:
# Step 7 & 8 & 9: Rock-Solid Native PyTorch Training Loop with AMP
import time
import torch
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"[TRAINING] Starting Native PyTorch Training on {device} (Mixed Precision AMP: {torch.cuda.is_available()})...")
print(f"[TRAINING] Total Batches per Epoch: {len(train_loader)} | Validation Batches: {len(val_loader)}")

best_val_acc = 0.0
final_model_dir = os.path.join(OUTPUT_DIR, "factora_image_model")
os.makedirs(final_model_dir, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    start_t = time.time()
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if (step + 1) % 150 == 0 or (step + 1) == len(train_loader):
            avg_loss = total_loss / (step + 1)
            acc = (correct / total) * 100
            print(f"Epoch [{epoch}/{NUM_EPOCHS}] Step [{step + 1}/{len(train_loader)}] | Loss: {avg_loss:.4f} | Train Acc: {acc:.2f}%")
            
    # Validation Loop
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(pixel_values=pixel_values, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_acc = (val_correct / val_total) * 100
    elapsed = time.time() - start_t
    print(f"\n>>> Epoch {epoch} Complete in {elapsed:.1f}s | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss / len(val_loader):.4f}\n")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"[CHECKPOINT] New Best Accuracy: {val_acc:.2f}%. Saving to {final_model_dir}...")
        model.save_pretrained(final_model_dir)
        processor.save_pretrained(final_model_dir)

print(f"\n[COMPLETE] Training Finished! Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Model files saved in: {final_model_dir}")
for f in os.listdir(final_model_dir):
    fpath = os.path.join(final_model_dir, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  - {f} ({size_mb:.2f} MB)")


In [ ]:
# Step 12: Interactive Single-Image Forensic Test
def verify_test_image(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(img, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    
    pred_idx = np.argmax(probs)
    verdict = ID2LABEL[pred_idx]
    confidence = float(probs[pred_idx]) * 100
    print(f"VERDICT: {verdict} | CONFIDENCE: {confidence:.2f}% | (Real: {probs[0]*100:.1f}%, Fake: {probs[1]*100:.1f}%)")

# Test on sample image if available
if len(val_dataset) > 0:
    sample_path = val_dataset.image_paths[0]
    verify_test_image(sample_path)